# 項目：整理Netflix電影演員評分數據

## 分析目標

此數據分析的目的是，整理不同流派影視作品，比如喜劇片、動作片、科幻片中，各演員出演作品的平均IMDB評分，從而挖掘出各個流派中的高評分作品演員。

本實戰項目的目的在於練習整理數據，從而得到可供下一步分析的數據。

## 簡介

原始資料集記錄了截至 2022 年 7 月 美國地區可觀看的所有 Netflix 電視劇及電影數據。資料集包含兩個資料表：'titles.csv' 和 'credits.csv'。
titles.csv 包含電影及電視劇相關資訊，包括影視作品 ID、標題、類型、描述、流派、IMDB（一個國外的線上評分網站）評分，等等。
credits.csv 包含超過 7 萬名 出現在 Netflix 影視作品的導演及演員資訊，包括名字、影視作品 ID、人物名、演職人員類型（導演/演員）等。

`titles.csv`每列的含義如下：
- id：影視作品ID。
- title：影視作品標題。
- show_type：作品類型，電視節目或電影。
- description：簡短描述。
- release_year：發布年份。
- age_certification：適齡認證。
- runtime：每集電視劇或電影的長度。
- genres：流派類型列表。
- production_countries：出品國家列表。
- seasons：如果是電視劇，則是季數。
- imdb_id：IMDB的ID。
- imdb_score：IMDB的評分。
- imdb_votes：IMDB的投票數。
- tmdb_popularity：TMDB的流行度。
- tmdb_score：TMDB的評分。

`credits.csv`每列的含义如下：
- person_ID：演職人員 ID。
- id：參與的影視作品ID。
- name：姓名。
- character_name：角色姓名。
- role：演職人員類型，演員或導演。

In [1]:
import pandas as pd

In [2]:
original_titles = pd.read_csv('titles.csv')
original_credits = pd.read_csv('credits.csv')

In [3]:
original_titles.head()

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600


In [4]:
original_credits.head()

,person_id,id,name,character,role
0,3748,tm84618,Robert De Niro,Travis Bickle,ACTOR
1,14658,tm84618,Jodie Foster,Iris Steensma,ACTOR
2,7064,tm84618,Albert Brooks,Tom,ACTOR
3,3739,tm84618,Harvey Keitel,Matthew 'Sport' Higgins,ACTOR
4,48933,tm84618,Cybill Shepherd,Betsy,ACTOR


先進行數據清理及評估

In [5]:
cleaned_titles = original_titles.copy()
cleaned_credits = original_credits.copy()

In [6]:
cleaned_titles.head()

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600


genre及production_country有多個值，應進行拆分

In [7]:
cleaned_titles['genres'][1]

"['drama', 'crime']"

雖然genres表示形式是列表，但其實際類型並非字串列表，而是字串，無法直接用value_counts統計各個值出現的次數。 我們可以使用 Python 內建的 eval函數，它可以把字串轉換成表達式，所以可以幫我們把表示列表的字串轉換成列表本身。

In [8]:
cleaned_titles['genres'] = cleaned_titles['genres'].apply(lambda s: eval(s))
cleaned_titles['genres'][1]

['drama', 'crime']

轉換為列表後，就能用 DataFrame的explode方法，把那一列的列表值拆分成單獨的行。

In [9]:
cleaned_titles = cleaned_titles.explode('genres')
cleaned_titles.head(10)

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,documentation,['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,drama,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,crime,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,drama,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,action,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,thriller,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,european,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,fantasy,['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,action,['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,comedy,['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811


接下來檢查production_countries

In [10]:
cleaned_titles['production_countries'][1]

1    ['US']
1    ['US']
Name: production_countries, dtype: object

可以看到production_countries已是字串，因此不須用eval函數。

In [11]:
cleaned_titles = cleaned_titles.explode('production_countries')
cleaned_titles.head(10)

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,documentation,['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,drama,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,crime,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,drama,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,action,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,thriller,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,european,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,fantasy,['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,action,['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,comedy,['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811


處理完cleaned_titles後，檢查cleaned_credits

In [12]:
cleaned_credits.head(10)

,person_id,id,name,character,role
0,3748,tm84618,Robert De Niro,Travis Bickle,ACTOR
1,14658,tm84618,Jodie Foster,Iris Steensma,ACTOR
2,7064,tm84618,Albert Brooks,Tom,ACTOR
3,3739,tm84618,Harvey Keitel,Matthew 'Sport' Higgins,ACTOR
4,48933,tm84618,Cybill Shepherd,Betsy,ACTOR
5,32267,tm84618,Peter Boyle,Wizard,ACTOR
6,519612,tm84618,Leonard Harris,Senator Charles Palantine,ACTOR
7,29068,tm84618,Diahnne Abbott,Concession Girl,ACTOR
8,519613,tm84618,Gino Ardito,Policeman at Rally,ACTOR
9,3308,tm84618,Martin Scorsese,Passenger Watching Silhouette,ACTOR


從頭部的10行數據來看，cleaned_credits 數據符合「每個變數為一列，每個觀察值為一行，每種類型的觀察單位為一個表格」，因此不存在結構性問題。

數據乾淨度

接下來透過info, 對數據內容進行大致了解

In [13]:
cleaned_titles.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 15147 entries, 0 to 5849
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    15147 non-null  object 
 1   title                 15146 non-null  object 
 2   type                  15147 non-null  object 
 3   description           15125 non-null  object 
 4   release_year          15147 non-null  int64  
 5   age_certification     9298 non-null   object 
 6   runtime               15147 non-null  int64  
 7   genres                15088 non-null  object 
 8   production_countries  15147 non-null  object 
 9   seasons               5923 non-null   float64
 10  imdb_id               14525 non-null  object 
 11  imdb_score            14399 non-null  float64
 12  imdb_votes            14375 non-null  float64
 13  tmdb_popularity       14995 non-null  float64
 14  tmdb_score            14614 non-null  float64
dtypes: float64(5), int64

從輸出結果來看，cleaned_titles 數據共有 17818 條觀察值，title、description、age_certification、genres、production_countries、seasons、imdb_id、imdb_score、tmdb_popularity、tmdb_score、imdb_votes、tmdb_popularity、tmdb_score 變數均存在缺失值，將在後續進行評估和清理。

此外，release_year 表示年份，數據類型不應為數字，應為日期，所以需要進行數據格式轉換（已於下一行先行轉換）。

In [14]:
cleaned_titles['release_year'] = pd.to_datetime(cleaned_titles['release_year'],format='%Y')
cleaned_titles['release_year']

0      1945-01-01
1      1976-01-01
1      1976-01-01
2      1972-01-01
2      1972-01-01
          ...    
5847   2021-01-01
5848   2021-01-01
5849   2021-01-01
5849   2021-01-01
5849   2021-01-01
Name: release_year, Length: 15147, dtype: datetime64[ns]

In [15]:
cleaned_credits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77801 entries, 0 to 77800
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   person_id  77801 non-null  int64 
 1   id         77801 non-null  object
 2   name       77801 non-null  object
 3   character  68029 non-null  object
 4   role       77801 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.0+ MB


從輸出結果來看，cleaned_credits 數據共有 77801 條觀察值，其中 character 變數存在缺失值，將在後續進行評估和清理。

此外，person_id 表示演職人員 ID，數據類型不應為數字，應為字串，所以需要進行數據格式轉換。

In [16]:
cleaned_credits['person_id']=cleaned_credits['person_id'].astype('str')
cleaned_credits['person_id']

0           3748
1          14658
2           7064
3           3739
4          48933
          ...   
77796     736339
77797     399499
77798     373198
77799     378132
77800    1950416
Name: person_id, Length: 77801, dtype: object

在 cleaned_titles 中，title、description、age_certification、genres、production_countries、seasons、imdb_id、imdb_score、tmdb_popularity、tmdb_score、imdb_votes、tmdb_popularity、tmdb_score 變數存在缺失值。但只有 imdb_score 和 genres，即 IMDB 評分和流派，與我們後續要做的分析息息相關。

先提取出 imdb_score 缺失的觀察值進行查看。

In [17]:
cleaned_titles[cleaned_titles['imdb_score'].isnull()]

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945-01-01,TV-MA,51,documentation,['US'],1.0,NaN,NaN,NaN,0.600,NaN
75,tm132164,Bill Hicks: Sane Man,MOVIE,Sane Man was filmed before Bill recorded ‘Dang...,1989-01-01,R,80,comedy,['US'],NaN,NaN,NaN,NaN,3.377,7.5
145,ts251477,My First Errand,SHOW,“Hajimete no Otsukai” (First Errand) is a Japa...,1991-01-01,TV-G,18,documentation,['JP'],12.0,NaN,NaN,NaN,7.730,7.8
145,ts251477,My First Errand,SHOW,“Hajimete no Otsukai” (First Errand) is a Japa...,1991-01-01,TV-G,18,family,['JP'],12.0,NaN,NaN,NaN,7.730,7.8
145,ts251477,My First Errand,SHOW,“Hajimete no Otsukai” (First Errand) is a Japa...,1991-01-01,TV-G,18,reality,['JP'],12.0,NaN,NaN,NaN,7.730,7.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5810,tm1225897,Social Man,MOVIE,Two competitive social media Influencers go he...,2021-01-01,NaN,96,drama,[],NaN,tt20198164,NaN,NaN,NaN,NaN
5833,ts307884,HQ Barbers,SHOW,When a family run barber shop in the heart of ...,2021-01-01,TV-14,24,comedy,['NG'],1.0,NaN,NaN,NaN,0.840,NaN
5840,tm1216735,Sun of the Soil,MOVIE,"In 14th-century Mali, an ambitious young royal...",2022-01-01,NaN,26,NaN,[],NaN,NaN,NaN,NaN,1.179,7.0
5844,tm1074617,Bling Empire - The Afterparty,MOVIE,"The stars of ""Bling Empire"" discuss the show's...",2021-01-01,NaN,35,NaN,['US'],NaN,NaN,NaN,NaN,NaN,NaN


由於缺失分析所需的核心數據 imdb_score，我們將把這些觀察值刪除，並查看刪除後該列空缺值個數

In [18]:
cleaned_titles = cleaned_titles.dropna(subset=['imdb_score'])
cleaned_titles["imdb_score"].isnull().sum()

0

In [19]:
cleaned_titles[cleaned_titles['genres'].isnull()]

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
1813,ts77824,My Next Guest Needs No Introduction With David...,SHOW,TV legend David Letterman teams up with fascin...,2018-01-01,TV-MA,50,NaN,['US'],4.0,tt7829834,7.8,5581.0,8.217,7.6
1939,ts215037,Minecraft: Story Mode,SHOW,"MInecraft: Story Mode is an interactive, anima...",2018-01-01,TV-PG,52,NaN,['US'],1.0,tt10498322,5.6,347.0,NaN,NaN
2386,ts74805,A Little Help with Carol Burnett,SHOW,In this unscripted series starring comedy lege...,2018-01-01,TV-G,24,NaN,['US'],1.0,tt7204366,6.3,237.0,1.621,6.2
2658,ts265844,#ABtalks,SHOW,#ABtalks is a YouTube interview show hosted by...,2018-01-01,TV-PG,68,NaN,[],1.0,tt12635254,9.6,7.0,NaN,NaN
4274,tm1172010,The Lockdown Plan,MOVIE,NaN,2020-01-01,NaN,49,NaN,[],NaN,tt13079112,6.5,NaN,NaN,NaN
4648,tm1113921,In Vitro,MOVIE,'In Vitro' is an otherworldly rumination on me...,2019-01-01,NaN,27,NaN,[],NaN,tt10545994,7.7,NaN,NaN,NaN


In [20]:
cleaned_titles = cleaned_titles.dropna(subset=['genres'])
cleaned_titles["genres"].isnull().sum()

0

接下來評估cleaned_credits的缺失數據，其中只有character變數存在缺失值。

角色名並不影響我們挖掘各個流派中的高 IMDB 評分作品演員，並且此變數缺失也有可能因為演職人員類別是導演，沒有對應角色，因此可以保留character 變數值存在空缺的觀察值。

處理重複數據

In [21]:
cleaned_titles.duplicated().sum()

0

In [22]:
cleaned_credits.duplicated().sum()

0

處理不一致數據

針對 cleaned_titles，不一致數據可能存在於 genres 和 production_countries 變數中，我們將查看是否存在多個不同值指代同一流派，以及多個不同值指代同一國家的情況。

In [23]:
cleaned_titles['genres'].value_counts()

drama            2827
comedy           2218
thriller         1180
action           1109
romance           955
crime             909
documentation     859
family            652
animation         630
fantasy           619
scifi             560
european          428
horror            366
history           254
music             244
reality           219
sport             170
war               155
western            39
Name: genres, dtype: int64

In [24]:
cleaned_titles[cleaned_titles['genres']!=""]
cleaned_titles.query('genres == ""')

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score


接下來，針對 production_countries 列也是一樣的流程，利用 value_counts 方法，得到 production_countries 的列表裡面各個值的出現次數。

In [25]:
cleaned_titles['production_countries'].value_counts()

['US']                4587
['IN']                1545
['JP']                 985
['KR']                 618
['GB']                 554
                      ... 
['CA', 'FR', 'LB']       1
['PH', 'US']             1
['CA', 'HU', 'US']       1
['Lebanon']              1
['PE', 'DE', 'NO']       1
Name: production_countries, Length: 427, dtype: int64

由於 value_counts 執行結果中有太多值，Pandas 只會預設顯示開頭和結尾的一些值。要完整展示結果，可以把 display.max_rows 設定為 None，即取消展示行數上限。

但因為我們只是在當前調用 value_counts 時才需要看完整結果，所以可以結合 option_context，只更改臨時上限。

In [27]:
with pd.option_context('display.max_rows', None):
    print(cleaned_titles['production_countries'].value_counts())

['US']                                        4587
['IN']                                        1545
['JP']                                         985
['KR']                                         618
['GB']                                         554
['ES']                                         466
['FR']                                         348
[]                                             300
['CA']                                         255
['MX']                                         198
['TR']                                         192
['BR']                                         186
['PH']                                         180
['DE']                                         179
['CN']                                         164
['AU']                                         150
['ID']                                         143
['IT']                                         139
['CA', 'US']                                   127
['GB', 'US']                   

從以上輸出結果來看，出品國家都用兩位的國家代碼來表示，除了裡面存在一個 Lebanon 值。
Lebanon 的國家代碼是 LB，出現了 39 次，說明此處數據不一致。LB 和 Lebanon 都在表示同一國家，需要進行統一。
把 cleaned_titles 裡，production_countries 的 "LB" 和 "Lebanon" 統一為 LB，並檢查替換後是否還存在 "LB"：

In [32]:
# 統一為 LB
cleaned_titles['production_countries'] = cleaned_titles['production_countries'].replace({'Lebanon':'LB'})
# 檢查Lebanon是否還存在
with pd.option_context('display.max_rows', None):
    print(cleaned_titles.explode('production_countries')['production_countries'].value_counts())


['US']                                        4587
['IN']                                        1545
['JP']                                         985
['KR']                                         618
['GB']                                         554
['ES']                                         466
['FR']                                         348
[]                                             300
['CA']                                         255
['MX']                                         198
['TR']                                         192
['BR']                                         186
['PH']                                         180
['DE']                                         179
['CN']                                         164
['AU']                                         150
['ID']                                         143
['IT']                                         139
['CA', 'US']                                   127
['GB', 'US']                   

In [33]:
original_credits['role'].value_counts()

ACTOR       73251
DIRECTOR     4550
Name: role, dtype: int64

從以上輸出結果來看，role 只有兩種可能的值，ACTOR 或 DIRECTOR，不存在不一致數據。我們可以把這列的類型轉換為 Category，好處是比字串類型更節約記憶體空間，也能表明說值的類型有限。

In [35]:
cleaned_credits['role'] = cleaned_credits['role'].astype('category')
cleaned_credits['role']

0           ACTOR
1           ACTOR
2           ACTOR
3           ACTOR
4           ACTOR
           ...   
77796       ACTOR
77797       ACTOR
77798       ACTOR
77799       ACTOR
77800    DIRECTOR
Name: role, Length: 77801, dtype: category
Categories (2, object): ['ACTOR', 'DIRECTOR']

In [36]:
original_titles.describe()

,release_year,runtime,seasons,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
count,5850.000000,5850.000000,2106.000000,5368.000000,5.352000e+03,5759.000000,5539.000000
mean,2016.417094,76.888889,2.162868,6.510861,2.343938e+04,22.637925,6.829175
std,6.937726,39.002509,2.689041,1.163826,9.582047e+04,81.680263,1.170391
min,1945.000000,0.000000,1.000000,1.500000,5.000000e+00,0.009442,0.500000
25%,2016.000000,44.000000,1.000000,5.800000,5.167500e+02,2.728500,6.100000
50%,2018.000000,83.000000,1.000000,6.600000,2.233500e+03,6.821000,6.900000
75%,2020.000000,104.000000,2.000000,7.300000,9.494000e+03,16.590000,7.537500
max,2022.000000,240.000000,42.000000,9.600000,2.294231e+06,2274.044000,10.000000


從以上統計資訊來看，original_titles 裡不存在脫離現實意義的數值。
original_credits 由於不包含表示數值含義的變數，因此無需用 describe 檢查。

In [37]:
cleaned_titles

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,drama,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,crime,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972-01-01,R,109,drama,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972-01-01,R,109,action,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972-01-01,R,109,thriller,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5846,tm898842,C/O Kaadhal,MOVIE,A heart warming film that explores the concept...,2021-01-01,NaN,134,drama,[],NaN,tt11803618,7.7,348.0,NaN,NaN
5847,tm1059008,Lokillo,MOVIE,A controversial TV host and comedian who has b...,2021-01-01,NaN,90,comedy,['CO'],NaN,tt14585902,3.8,68.0,26.005,6.300
5849,ts271048,Mighty Little Bheem: Kite Festival,SHOW,"With winter behind them, Bheem and his townspe...",2021-01-01,NaN,7,family,[],1.0,tt13711094,7.8,18.0,2.289,10.000
5849,ts271048,Mighty Little Bheem: Kite Festival,SHOW,"With winter behind them, Bheem and his townspe...",2021-01-01,NaN,7,animation,[],1.0,tt13711094,7.8,18.0,2.289,10.000


In [38]:
cleaned_credits

,person_id,id,name,character,role
0,3748,tm84618,Robert De Niro,Travis Bickle,ACTOR
1,14658,tm84618,Jodie Foster,Iris Steensma,ACTOR
2,7064,tm84618,Albert Brooks,Tom,ACTOR
3,3739,tm84618,Harvey Keitel,Matthew 'Sport' Higgins,ACTOR
4,48933,tm84618,Cybill Shepherd,Betsy,ACTOR
...,...,...,...,...,...
77796,736339,tm1059008,Adelaida Buscato,María Paz,ACTOR
77797,399499,tm1059008,Luz Stella Luengas,Karen Bayona,ACTOR
77798,373198,tm1059008,Inés Prieto,Fanny,ACTOR
77799,378132,tm1059008,Isabel Gaona,Cacica,ACTOR


In [52]:
credits_with_titles = pd.merge(cleaned_titles, cleaned_credits, on = 'id', how = 'inner')
credits_with_titles.head(5)

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,person_id,name,character,role
0,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,drama,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179,3748,Robert De Niro,Travis Bickle,ACTOR
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,drama,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179,14658,Jodie Foster,Iris Steensma,ACTOR
2,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,drama,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179,7064,Albert Brooks,Tom,ACTOR
3,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,drama,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179,3739,Harvey Keitel,Matthew 'Sport' Higgins,ACTOR
4,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,drama,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179,48933,Cybill Shepherd,Betsy,ACTOR


為了能同時獲得流派與演員數據，我們需要把 cleaned_titles 和 cleaned_credits，通過 id 作為鍵進行連接，因為兩個資料表中 id 都是影視作品 ID。

In [49]:
actor_with_titles = credits_with_titles.query('role == "ACTOR"')

為了挖掘出各個流派中的高 IMDB 評分作品演員，我們需要先根據流派和演員進行分組。

對演員進行分組的時候，我們用 person_id 而不是 name 變數，原因是名字容易出現拼錯或者重名的情況，演職人員 ID 會比演員姓名更加準確地反映是哪位演員。

In [51]:
group_genres_and_actor = actor_with_titles.groupby(['genres','person_id'])

分組後，我們只需要對 imdb_score 的值進行聚合計算，因此只提取 imdb_score 變數，然後調用 mean，來計算各個流派影視作品中，每位演員參演作品的平均 IMDB 評分。

In [59]:
imdb_score_groupby_genres_and_person_id = group_genres_and_actor['imdb_score'].mean()
imdb_score_groupby_genres_and_person_id

genres   person_id
action   1000         6.866667
         100007       7.000000
         100013       6.400000
         100019       6.500000
         100020       6.500000
                        ...   
western  993735       6.500000
         998673       7.300000
         998674       7.300000
         998675       7.300000
         99940        4.000000
Name: imdb_score, Length: 168881, dtype: float64

In [64]:
imdb_score_groupby_genres_and_person_id_df = imdb_score_groupby_genres_and_person_id.reset_index()
imdb_score_groupby_genres_and_person_id_df

,genres,person_id,imdb_score
0,action,1000,6.866667
1,action,100007,7.000000
2,action,100013,6.400000
3,action,100019,6.500000
4,action,100020,6.500000
...,...,...,...
168876,western,993735,6.500000
168877,western,998673,7.300000
168878,western,998674,7.300000
168879,western,998675,7.300000


現在針對流派和演員分組的 IMDB 評分數據已經整理好，可以進入後續的分析步驟了。

但我們當前可以繼續做一些數據整理，比如對上面的結果再次進行分組，找出各個流派裡演員作品最高的平均評分是多少、最高評分對應的演員名字是什麼，
要得到這一結果，我們需要再次用 genres 進行分組，然後提取出 imdb_score 變數，計算其最大值。

In [66]:
genres_max_score = imdb_score_groupby_genres_and_person_id_df.groupby('genres')['imdb_score'].max()
genres_max_score

genres
action           9.3
animation        9.3
comedy           9.2
crime            9.5
documentation    9.1
drama            9.5
european         8.9
family           9.3
fantasy          9.3
history          9.1
horror           9.0
music            8.8
reality          8.9
romance          9.2
scifi            9.3
sport            9.1
thriller         9.5
war              8.8
western          8.9
Name: imdb_score, dtype: float64

在我們知道最高分後，可以把以上結果和之前得到的 imdb_score_groupby_genres_and_person_id_df 再次進行連接，得到最高分對應的各個演員 ID 是什麼，也就是這個最高平均分是哪位演員拿到的。

In [70]:
genre_max_score_imdb_score_groupby_genres_and_person_id_df = pd.merge(imdb_score_groupby_genres_and_person_id_df,genres_max_score,on = ['genres','imdb_score'])
genre_max_score_imdb_score_groupby_genres_and_person_id_df

,genres,person_id,imdb_score
0,action,12790,9.3
1,action,1303,9.3
2,action,21033,9.3
3,action,336830,9.3
4,action,86591,9.3
...,...,...,...
131,war,826547,8.8
132,western,22311,8.9
133,western,28166,8.9
134,western,28180,8.9


從以上的結果可以看出，最高分所對應的演員不一定只有一位，可能有多位演員的平均得分相同。

為了得到演員ID所對應的演員名字，我們可以和 cleaned_credits 這個 DataFrame 進行連接（Join）。這個 DataFrame 還有其他列，我們只需要得到 person_id 和 name 的對應，所以可以先提取出那兩列，並把重複的行刪除。

In [73]:
actor_id_name = cleaned_credits[['person_id','name']].drop_duplicates()
actor_id_name.head(5)

,person_id,name
0,3748,Robert De Niro
1,14658,Jodie Foster
2,7064,Albert Brooks
3,3739,Harvey Keitel
4,48933,Cybill Shepherd


In [76]:
genre_max_score_with_actor_name = pd.merge(genre_max_score_imdb_score_groupby_genres_and_person_id_df, actor_id_name, on ='person_id')
genre_max_score_with_actor_name

,genres,person_id,imdb_score,name
0,action,12790,9.3,Olivia Hack
1,scifi,12790,9.3,Olivia Hack
2,action,1303,9.3,Jessie Flower
3,animation,1303,9.3,Jessie Flower
4,family,1303,9.3,Jessie Flower
...,...,...,...,...
131,war,826547,8.8,Yuto Uemura
132,western,22311,8.9,Koichi Yamadera
133,western,28166,8.9,Megumi Hayashibara
134,western,28180,8.9,Unsho Ishizuka


為了把相同流派都排序在一起，我們還可以用 sort_values 方法，根據結果中的行，依照流派（genres）進行排序，然後用 reset_index 把索引重新排序。

索引重新排序後，DataFrame 會多出一個 index 欄位（列），我們可以再把這個 index 欄位進行刪除。

In [88]:
genre_max_score_with_actor_name = genre_max_score_with_actor_name.sort_values('genres').drop(['index','level_0'],axis = 1)
genre_max_score_with_actor_name

,genres,person_id,imdb_score,name
0,action,12790,9.3,Olivia Hack
1,action,336830,9.3,André Sogliuzzo
2,action,21033,9.3,Zach Tyler
3,action,86591,9.3,Cricket Leigh
4,action,1303,9.3,Jessie Flower
...,...,...,...,...
131,war,826547,8.8,Yuto Uemura
133,western,28166,8.9,Megumi Hayashibara
134,western,28180,8.9,Unsho Ishizuka
132,western,22311,8.9,Koichi Yamadera
